In [1]:
# Import Libraries and Connect to AWS S3
import s3fs
fs = s3fs.S3FileSystem(anon=True)
import pandas as pd
pd.set_option('display.max_columns', None)
import ast
import re

In [2]:
# Load the Slurm Log
df_slurm = pd.read_csv('s3://mit-supercloud-dataset/datacenter-challenge/202201/slurm-log.csv')
print(df_slurm.shape)

(395914, 29)


In [3]:
# Create new columns for job duration
df_slurm['duration_seconds'] = df_slurm['time_end'] - df_slurm['time_start']
df_slurm['duration_hours'] = df_slurm['duration_seconds'] / 3600

In [4]:
# Create new column for count of GPU's allocated for each job
def get_allocated_gpu_count(tres_str):
    tres_str = str(tres_str)
    mappings = dict(re.findall(r'(\d+)=(\d+)', tres_str))
    
    # Priority 1: Check MIT Specific IDs (1001 or 1002)
    for specific_id in ['1001', '1002']:
        if specific_id in mappings:
            return int(mappings[specific_id])
    return 0

df_slurm['gpus_alloc'] = df_slurm['tres_alloc'].apply(get_allocated_gpu_count)
df_slurm[['time_start', 'time_end', 'duration_seconds', 'duration_hours', 'tres_alloc', 'gpus_alloc']].head(1)

,time_start,time_end,duration_seconds,duration_hours,tres_alloc,gpus_alloc
0,1609806297,1609806605,308,0.085556,"1=20,2=170000,4=1,5=20",0


In [5]:
# Create subset of jobs that were over 2 hours in duration which also only ran on 1 Node
df_slurm_2hr = df_slurm[(df_slurm['duration_hours'] > 2) & (df_slurm['nodes_alloc'] == 1)]
print(df_slurm_2hr.shape)

(126532, 32)


#
# Gathering Example Jobs from the ">2 Hour" Subset:
#### I want to look at the CPU and GPU timeseries logs for jobs that fall into different categories
#### I'm looking at the actual data to brainstorm new features to engineer that could potentially predict job failures

In [ ]:
# Category 1.1 --  End Job State = "3- Completed" AND 0 GPU's allocated
# Category 1.2 --  End Job State = "5- Failed (Crashed)" AND 0 GPU's allocated
# Category 1.3 --  End Job State = "6- Failed (Timeout)" AND 0 GPU's allocated
# Category 1.4 --  End Job State = "11- Failed (Out of Memory)" AND 0 GPU's allocated

# Category 2.1 --  End Job State = "3- Completed" AND 1 GPU allocated
# Category 2.2 --  End Job State = "5- Failed (Crashed)" AND 1 GPU allocated
# Category 2.3 --  End Job State = "6- Failed (Timeout)" AND 1 GPU allocated
# Category 2.4 --  End Job State = "11- Failed (Out of Memory)" AND 1 GPU allocated

# Category 3.1 --  End Job State = "3- Completed" AND 2 GPU's allocated
# Category 3.2 --  End Job State = "5- Failed (Crashed)" AND 2 GPU's allocated
# Category 3.3 --  End Job State = "6- Failed (Timeout)" AND 2 GPU's allocated
# Category 3.4 --  End Job State = "11- Failed (Out of Memory)" AND 2 GPU's allocated

In [6]:
# Finding a job in Category 1.1
df_1_1 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 0) & (df_slurm_2hr['state'] == 3)]

# We use .sample(1) to get a random entry
sample_row_1_1 = df_1_1.sample(1)
sample_row_1_1

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
195508,49907319995280,16618712154521,4294967294,5577359963038,61026541062099,1,['r6272977-n680758'],4,0,0,NaN,0,0,\N,8,9223372036854779608,xeon-p8,10007,3,4300,1620816372,1620816372,1620820242,1620827789,0,0,"1=4,2=15200,4=1,5=4","1=4,2=15200,4=1,5=4",OTHER,7547,2.096389,0


In [7]:
# Gathering CPU Logs for this job by searching the MIT Supercloud directory for a file with a matching job ID
job_id = '49907319995280'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 49907319995280.
Created variable: job_49907319995280_CPUSummary_1
Created variable: job_49907319995280_CPUTimeSeries_1


In [8]:
# Gathering GPU Log for this job (if it exists)
job_id = '49907319995280'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 0 GPU log file(s) for Job ID 49907319995280.
No GPU files found for Job 49907319995280.


In [61]:
# Save Logs to CSV files for analysis
job_49907319995280_CPUSummary_1.to_csv('CPU_Summary_1.1.csv', index=False)
job_49907319995280_CPUTimeSeries_1.to_csv('CPU_Timeseries_1.1.csv', index=False)

#
# Doing the same process for the rest of the categories
### I probably could have used a loop, but decided to just copy/paste since there were only 11 more categories

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 1.2                          #
#                                                                              #
################################################################################

In [63]:
# Finding a job in Category 1.2
df_1_2 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 0) & (df_slurm_2hr['state'] == 5)]
sample_row_1_2 = df_1_2.sample(1)
sample_row_1_2

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
54300,39673954616818,20944089809881,8,12798940487396,61026541062099,1,['r5573787-n386398'],1,0,256,NaN,0,0,xeon-g6,4,20480,normal,10450,5,4294967295,1612801946,1612801947,1612801947,1612810443,0,0,"1=1,2=20480,4=1,5=1","1=1,2=20480,4=1,5=1",OTHER,8496,2.36,0


In [65]:
# Gathering CPU Logs
job_id = '39673954616818'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 39673954616818.
Created variable: job_39673954616818_CPUSummary_1
Created variable: job_39673954616818_CPUTimeSeries_1


In [67]:
# Gathering GPU Log (if it exists)
job_id = '39673954616818'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 0 GPU log file(s) for Job ID 39673954616818.
No GPU files found for Job 39673954616818.


In [103]:
# Save Logs to CSV files for analysis
job_39673954616818_CPUSummary_1.to_csv('CPU_Summary_1.2.csv', index=False)
job_39673954616818_CPUTimeSeries_1.to_csv('CPU_Timeseries_1.2.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 1.3                          #
#                                                                              #
################################################################################

In [105]:
# Finding a job in Category 1.3
df_1_3 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 0) & (df_slurm_2hr['state'] == 6)]
sample_row_1_3 = df_1_3.sample(1)
sample_row_1_3

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
56980,13071821567838,16618712154521,4294967294,40938513184947,61026541062099,1,['r2652301-n911952'],4,0,0,NaN,0,0,xeon-g6,4,9223372036854784308,normal,110379,6,720,1613571248,1613571248,1613571248,1613614473,0,0,"1=4,2=34000,4=1,5=4","1=4,2=34000,4=1,5=4",OTHER,43225,12.006944,0


In [107]:
# Gathering CPU Logs
job_id = '13071821567838'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 13071821567838.
Created variable: job_13071821567838_CPUSummary_1
Created variable: job_13071821567838_CPUTimeSeries_1


In [109]:
# Gathering GPU Log (if it exists)
job_id = '13071821567838'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 0 GPU log file(s) for Job ID 13071821567838.
No GPU files found for Job 13071821567838.


In [113]:
# Save Logs to CSV files for analysis
job_13071821567838_CPUSummary_1.to_csv('CPU_Summary_1.3.csv', index=False)
job_13071821567838_CPUTimeSeries_1.to_csv('CPU_Timeseries_1.3.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 1.4                          #
#                                                                              #
################################################################################

In [115]:
# Finding a job in Category 1.4
df_1_4 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 0) & (df_slurm_2hr['state'] == 11)]
sample_row_1_4 = df_1_4.sample(1)
sample_row_1_4

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
195027,90819607424694,46348818171126,193,36880203429568,61026541062099,1,['r9192091-n680758'],1,0,253,NaN,0,0,\N,8,9223372036854779808,xeon-p8,10152,11,525600,1620791503,1620791504,1620791513,1620930064,0,0,"1=1,2=4000,4=1,5=1","1=1,2=4000,4=1,5=1",LLSUB:BATCH,138551,38.486389,0


In [117]:
# Gathering CPU Logs
job_id = '90819607424694'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 90819607424694.
Created variable: job_90819607424694_CPUSummary_1
Created variable: job_90819607424694_CPUTimeSeries_1


In [119]:
# Gathering GPU Log (if it exists)
job_id = '90819607424694'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 0 GPU log file(s) for Job ID 90819607424694.
No GPU files found for Job 90819607424694.


In [127]:
# Save Logs to CSV files for analysis
job_90819607424694_CPUSummary_1.to_csv('CPU_Summary_1.4.csv', index=False)
job_90819607424694_CPUTimeSeries_1.to_csv('CPU_Timeseries_1.4.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 2.1                          #
#                                                                              #
################################################################################

In [129]:
# Finding a job in Category 2.1
df_2_1 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 1) & (df_slurm_2hr['state'] == 3)]
sample_row_2_1 = df_2_1.sample(1)
sample_row_2_1

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
236300,4391237494359,16618712154521,4294967294,5042570016904,61026541062099,1,['r8937440-n43543'],20,0,0,NaN,0,0,\N,8,9223372036854784308,normal,10003,3,525600,1623428400,1623428400,1623532270,1623575066,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",OTHER,42796,11.887778,1


In [131]:
# Gathering CPU Logs
job_id = '4391237494359'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 4391237494359.
Created variable: job_4391237494359_CPUSummary_1
Created variable: job_4391237494359_CPUTimeSeries_1


In [133]:
# Gathering GPU Log (if it exists)
job_id = '4391237494359'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 4391237494359.
Created variable: job_4391237494359_GPULog_1


In [141]:
# Save Logs to CSV files for analysis
job_4391237494359_CPUSummary_1.to_csv('CPU_Summary_2.1.csv', index=False)
job_4391237494359_CPUTimeSeries_1.to_csv('CPU_Timeseries_2.1.csv', index=False)
job_4391237494359_GPULog_1.to_csv('GPU_Timeseries_2.1.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 2.2                          #
#                                                                              #
################################################################################

In [143]:
# Finding a job in Category 2.2
df_2_2 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 1) & (df_slurm_2hr['state'] == 5)]
sample_row_2_2 = df_2_2.sample(1)
sample_row_2_2

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
258812,80374537872607,16618712154521,4294967294,32808329113198,61026541062099,1,['r1682297-n976057'],20,0,256,NaN,0,0,\N,8,9223372036854784308,normal,10110,5,525600,1626649139,1626649139,1626649142,1626707158,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",OTHER,58016,16.115556,1


In [145]:
# Gathering CPU Logs
job_id = '80374537872607'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 80374537872607.
Created variable: job_80374537872607_CPUSummary_1
Created variable: job_80374537872607_CPUTimeSeries_1


In [149]:
# Gathering GPU Log (if it exists)
job_id = '80374537872607'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 80374537872607.
Created variable: job_80374537872607_GPULog_1


In [153]:
# Save Logs to CSV files for analysis
job_80374537872607_CPUSummary_1.to_csv('CPU_Summary_2.2.csv', index=False)
job_80374537872607_CPUTimeSeries_1.to_csv('CPU_Timeseries_2.2.csv', index=False)
job_80374537872607_GPULog_1.to_csv('GPU_Timeseries_2.2.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 2.3                          #
#                                                                              #
################################################################################

In [155]:
# Finding a job in Category 2.3
df_2_3 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 1) & (df_slurm_2hr['state'] == 6)]
sample_row_2_3 = df_2_3.sample(1)
sample_row_2_3

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
136889,65852109034682,16618712154521,4294967294,87544801505686,61026541062099,1,['r4858666-n830961'],20,0,0,NaN,0,0,xeon-g6,2,9223372036854784308,normal,110025,6,1440,1618781458,1618781458,1618781458,1618867869,0,0,"1=20,2=170000,4=1,5=20,1002=1","1=20,2=170000,4=1,5=20,1002=1",LLSUB:INTERACTIVE,86411,24.003056,1


In [157]:
# Gathering CPU Logs
job_id = '65852109034682'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 65852109034682.
Created variable: job_65852109034682_CPUSummary_1
Created variable: job_65852109034682_CPUTimeSeries_1


In [159]:
# Gathering GPU Log (if it exists)
job_id = '65852109034682'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 65852109034682.
Created variable: job_65852109034682_GPULog_1


In [167]:
# Save Logs to CSV files for analysis
job_65852109034682_CPUSummary_1.to_csv('CPU_Summary_2.3.csv', index=False)
job_65852109034682_CPUTimeSeries_1.to_csv('CPU_Timeseries_2.3.csv', index=False)
job_65852109034682_GPULog_1.to_csv('GPU_Timeseries_2.3.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 2.4                          #
#                                                                              #
################################################################################

In [169]:
# Finding a job in Category 2.4
df_2_4 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 1) & (df_slurm_2hr['state'] == 11)]
sample_row_2_4 = df_2_4.sample(1)
sample_row_2_4

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
73356,45239897292216,54277188975482,74,27491691635712,61026541062099,1,['r8015356-n976057'],1,0,253,NaN,0,0,xeon-g6,4,9223372036854779904,normal,110483,11,1440,1614739838,1614739838,1614783996,1614809789,0,0,"1=1,2=4096,4=1,5=1,1002=1","1=1,2=4096,4=1,5=1,1002=1",OTHER,25793,7.164722,1


In [171]:
# Gathering CPU Logs
job_id = '45239897292216'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 45239897292216.
Created variable: job_45239897292216_CPUSummary_1
Created variable: job_45239897292216_CPUTimeSeries_1


In [173]:
# Gathering GPU Log (if it exists)
job_id = '45239897292216'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 45239897292216.
Created variable: job_45239897292216_GPULog_1


In [181]:
# Save Logs to CSV files for analysis
job_45239897292216_CPUSummary_1.to_csv('CPU_Summary_2.4.csv', index=False)
job_45239897292216_CPUTimeSeries_1.to_csv('CPU_Timeseries_2.4.csv', index=False)
job_45239897292216_GPULog_1.to_csv('GPU_Timeseries_2.4.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 3.1                          #
#                                                                              #
################################################################################

In [189]:
# Finding a job in Category 3.1
df_3_1 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 2) & (df_slurm_2hr['state'] == 3)]
sample_row_3_1 = df_3_1.sample(1)
sample_row_3_1

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
243408,43725154320493,16618712154521,4294967294,64435430961080,61026541062099,1,['r1682297-n43543'],40,0,0,NaN,0,0,xeon-g6,4,9223372036854784308,normal,10150,3,2880,1624931466,1624931466,1624931467,1624968499,0,0,"1=40,2=340000,4=1,5=40,1002=2","1=40,2=340000,4=1,5=40,1002=2",LLSUB:BATCH,37032,10.286667,2


In [195]:
# Gathering CPU Logs
job_id = '43725154320493'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 43725154320493.
Created variable: job_43725154320493_CPUSummary_1
Created variable: job_43725154320493_CPUTimeSeries_1


In [191]:
# Gathering GPU Log (if it exists)
job_id = '43725154320493'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 43725154320493.
Created variable: job_43725154320493_GPULog_1


In [203]:
# Save Logs to CSV files for analysis
job_43725154320493_CPUSummary_1.to_csv('CPU_Summary_3.1.csv', index=False)
job_43725154320493_CPUTimeSeries_1.to_csv('CPU_Timeseries_3.1.csv', index=False)
job_43725154320493_GPULog_1.to_csv('GPU_Timeseries_3.1.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 3.2                          #
#                                                                              #
################################################################################

In [205]:
# Finding a job in Category 3.2
df_3_2 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 2) & (df_slurm_2hr['state'] == 5)]
sample_row_3_2 = df_3_2.sample(1)
sample_row_3_2

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
371948,69864988491646,16618712154521,4294967294,73955363208519,61026541062099,1,['r6272977-n976057'],1,0,256,NaN,0,0,xeon-g6,2,9223372036854784308,normal,110438,5,1440,1631997750,1631997750,1631997750,1632057976,0,0,"1=40,2=340000,4=1,5=40,1002=2","1=1,2=8500,4=1,5=1,1002=2",LLSUB:INTERACTIVE,60226,16.729444,2


In [207]:
# Gathering CPU Logs
job_id = '69864988491646'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 69864988491646.
Created variable: job_69864988491646_CPUSummary_1
Created variable: job_69864988491646_CPUTimeSeries_1


In [209]:
# Gathering GPU Log (if it exists)
job_id = '69864988491646'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 69864988491646.
Created variable: job_69864988491646_GPULog_1


In [217]:
# Save Logs to CSV files for analysis
job_69864988491646_CPUSummary_1.to_csv('CPU_Summary_3.2.csv', index=False)
job_69864988491646_CPUTimeSeries_1.to_csv('CPU_Timeseries_3.2.csv', index=False)
job_69864988491646_GPULog_1.to_csv('GPU_Timeseries_3.2.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 3.3                          #
#                                                                              #
################################################################################

In [11]:
# Finding a job in Category 3.3
df_3_3 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 2) & (df_slurm_2hr['state'] == 6)]
sample_row_3_3 = df_3_3.sample(1)
sample_row_3_3

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
273572,14570921812618,16618712154521,4294967294,72781180556911,61026541062099,1,['r1682297-n386398'],1,0,0,NaN,0,0,xeon-g6,4,9223372036854784308,normal,110063,6,1440,1627227443,1627227443,1627227443,1627313852,0,0,"1=40,2=340000,4=1,5=40,1002=2","1=1,2=8500,4=1,5=1,1002=2",OTHER,86409,24.0025,2


In [13]:
# Gathering CPU Logs
job_id = '14570921812618'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 14570921812618.
Created variable: job_14570921812618_CPUSummary_1
Created variable: job_14570921812618_CPUTimeSeries_1


In [15]:
# Gathering GPU Log (if it exists)
job_id = '14570921812618'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 14570921812618.
Created variable: job_14570921812618_GPULog_1


In [23]:
# Save Logs to CSV files for analysis
job_14570921812618_CPUSummary_1.to_csv('CPU_Summary_3.3.csv', index=False)
job_14570921812618_CPUTimeSeries_1.to_csv('CPU_Timeseries_3.3.csv', index=False)
job_14570921812618_GPULog_1.to_csv('GPU_Timeseries_3.3.csv', index=False)

In [ ]:
################################################################################
#                                                                              #
#                       Gathering Data : Category 3.4                          #
#                                                                              #
################################################################################

In [25]:
# Finding a job in Category 3.4
df_3_4 = df_slurm_2hr[(df_slurm_2hr['gpus_alloc'] == 2) & (df_slurm_2hr['state'] == 11)]
sample_row_3_4 = df_3_4.sample(1)
sample_row_3_4

,id_job,id_array_job,id_array_task,id_user,kill_requid,nodes_alloc,nodelist,cpus_req,derived_ec,exit_code,gres_used,array_max_tasks,array_task_pending,constraints,flags,mem_req,partition,priority,state,timelimit,time_submit,time_eligible,time_start,time_end,time_suspended,track_steps,tres_alloc,tres_req,job_type,duration_seconds,duration_hours,gpus_alloc
318901,75524799341295,26375623220140,2,77492909145526,61026541062099,1,['r3741709-n851693'],1,0,253,NaN,0,0,\N,4,9223372036854784308,normal,10142,11,4294967295,1630637015,1630637015,1630637015,1630670028,0,0,"1=1,2=8500,4=1,5=1,1002=2","1=1,2=8500,4=1,5=1,1002=2",OTHER,33013,9.170278,2


In [27]:
# Gathering CPU Logs
job_id = '75524799341295'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/cpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} total CPU log file(s) for Job ID {job_id}.")
if files:
    summary_count = 1
    ts_count = 1
    for file_path in files:
        full_path = f"s3://{file_path}"
        if 'summary' in file_path.lower():
            var_name = f"job_{job_id}_CPUSummary_{summary_count}"
            summary_count += 1
        else:
            var_name = f"job_{job_id}_CPUTimeSeries_{ts_count}"
            ts_count += 1
        globals()[var_name] = pd.read_csv(full_path)
        print(f"Created variable: {var_name}")
else:
    print(f"No .csv files found for Job ID {job_id}.")

Found 2 total CPU log file(s) for Job ID 75524799341295.
Created variable: job_75524799341295_CPUSummary_1
Created variable: job_75524799341295_CPUTimeSeries_1


In [29]:
# Gathering GPU Log (if it exists)
job_id = '75524799341295'
search_pattern = f"mit-supercloud-dataset/datacenter-challenge/202201/gpu/**/{job_id}*.csv"
files = fs.glob(search_pattern)
print(f"Found {len(files)} GPU log file(s) for Job ID {job_id}.")
if files:
    for i, file_path in enumerate(files, start=1):
        var_name = f"job_{job_id}_GPULog_{i}"
        globals()[var_name] = pd.read_csv(f"s3://{file_path}")
        print(f"Created variable: {var_name}")
else:
    print(f"No GPU files found for Job {job_id}.")

Found 1 GPU log file(s) for Job ID 75524799341295.
Created variable: job_75524799341295_GPULog_1


In [37]:
# Save Logs to CSV files for analysis
job_75524799341295_CPUSummary_1.to_csv('CPU_Summary_3.4.csv', index=False)
job_75524799341295_CPUTimeSeries_1.to_csv('CPU_Timeseries_3.4.csv', index=False)
job_75524799341295_GPULog_1.to_csv('GPU_Timeseries_3.4.csv', index=False)